# viewmaxxing: room video → walkable 3D (Gaussian splat + collision mesh)

**What you get** (in `/kaggle/working/output/`, downloadable as `viewmaxxing_<name>.zip`):

| File | Use it for |
|---|---|
| `scene.html` | Double-click it: a walk-around 3D viewer of your room in any browser |
| `scene.spz` | **Blender 5.3** (File → Import → PLY/SPZ), Scaniverse/SuperSplat, Godot add-on |
| `scene.sog` | Web engines (PlayCanvas, Spark/three.js), the smallest file |
| `scene.ply` | Raw splat, works everywhere (big) |
| `collision.glb` | Invisible low-poly walls and floor for physics / walking (import into Blender or Godot) |
| `align.json` | Spawn point, room bounds, scale info |

World frame: **+Y up, floor at y = 0, roughly metres**, walls lined up with the X/Z axes.

## Setup (once)
1. Right panel → **Settings → Accelerator → GPU T4 x2**. Don't pick P100: current PyTorch doesn't support it.
2. **Settings → Internet → On**. This needs a phone-verified Kaggle account.
3. **Add Input → Upload** your room video (or a folder of photos) as a new Dataset. Then put its path in `INPUT_PATH` below.
4. **Run All.** Expect about **25–40 min with `QUALITY="fast"`** and **60–90 min with `"full"`**. That uses about 1–1.5 of your 30 weekly GPU hours.

## How to film (this matters more than any setting)
- Use the **normal camera app in Video mode**, 1×, 4K or 1080p, with **HDR Video OFF** (Settings → Camera → Record Video).
- Light the room evenly. **Move slowly**, because blur kills it. Make two loops around the room: one at chest height looking outward, and one lower **pointing at the floor and furniture**.
- Spend 1–3 minutes per room. Cover every surface from 2–3 angles. Avoid filming mirrors and windows head-on.

In [ ]:
# ===== SETTINGS: edit these =====
INPUT_PATH      = "/kaggle/input/room-scan"  # a video file, a folder with one video, or a folder of photos
SCENE_NAME      = "my_room"
QUALITY         = "fast"     # "fast" (~7.5k steps) or "full" (30k steps, sharper, ~3x longer)
TARGET_FRAMES   = 200        # frames picked from the video (100-300). More = slower but more complete
MAX_IMAGE_SIZE  = 1600       # long side of frames in px. Lower (1280) if you run out of GPU memory
CAMERA_HEIGHT_M = 1.4        # how high you held the phone; used to guess real-world scale
KNOWN_SCALE     = None       # set a number to override the scale guess (see the scale note at the end)
SEED_POS        = None       # spawn/collider seed [x, y, z]; None = auto (middle of your walking path)
# =================================

In [ ]:
import os, sys, subprocess, json, shutil, glob, time
from pathlib import Path

W = Path("/kaggle/working"); VM = W / "vm"; DATA = W / "data"; RES = W / "results"; OUT = W / "output"
for p in (VM,): p.mkdir(parents=True, exist_ok=True)

def sh(cmd, cwd=None, env=None, tail=40):
    # Run a shell command, stream output, and fail loudly with the last lines.
    print(f"$ {cmd}")
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env={**os.environ, **(env or {})},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if not line.startswith(("I20", "W20")):  # hide COLMAP's chatty glog lines
            print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed ({p.returncode}). Last output:\n" + "".join(lines[-tail:]))
    print(f"  [done in {time.time() - t:.0f}s]")

sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv")
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU OK:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Set Settings > Accelerator > GPU T4 x2"
cc = torch.cuda.get_device_capability(0)
assert cc >= (7, 0), f"GPU compute capability {cc} is too old for current PyTorch. Pick GPU T4 x2, not P100."
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"  # compile CUDA only for this GPU (faster)

## 1. Install (≈10 min, mostly compiling gsplat's CUDA kernels once)

In [ ]:
# gsplat trainer (pinned release) and its example-script dependencies
sh("git clone -q --depth 1 --branch v1.5.3 https://github.com/nerfstudio-project/gsplat.git " + str(W / "gsplat")
   if not (W / "gsplat").exists() else "echo gsplat already cloned")
sh("pip install -q gsplat==1.5.3 viser tyro 'torchmetrics[image]' tensorly splines plyfile trimesh "
   "fast-simplification scikit-image imageio[ffmpeg] pillow-heif "
   "'git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787' "
   "'git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5'")

# gsplat's COLMAP reader is a small pure-python 'pycolmap' fork. It clashes by name with the official
# pycolmap (used for SfM), so the official one goes into its own folder and runs in a subprocess.
sh("pip install -q --no-deps 'git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e'")
import importlib.util
sm = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0]) / "scene_manager.py"
sm.write_text(sm.read_text().replace("np.uint64(-1)", "np.uint64(np.iinfo(np.uint64).max)"))  # numpy 2 fix

SFM_LIBS = W / "sfm_libs"
if not SFM_LIBS.exists():
    try:  # GPU build (fast SIFT + matching); fall back to CPU build if it won't load
        sh(f"pip install -q --target {SFM_LIBS} pycolmap-cuda12==4.2.1")
        sh(f"PYTHONPATH={SFM_LIBS} python -c 'import pycolmap; print(\"pycolmap\", pycolmap.__version__, \"CUDA:\", pycolmap.has_cuda)'")
    except RuntimeError as e:
        print("CUDA pycolmap unavailable, using CPU build:", str(e)[-300:])
        shutil.rmtree(SFM_LIBS, ignore_errors=True)
        sh(f"pip install -q --target {SFM_LIBS} pycolmap==4.2.1")

# Node.js + PlayCanvas splat-transform (compression and HTML viewer)
def node_ok():
    try:
        return int(subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip()[1:].split(".")[0]) >= 18
    except Exception:
        return False
if not node_ok():
    sh("curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz | tar -xJ -C /kaggle/working")
    os.environ["PATH"] = "/kaggle/working/node-v22.11.0-linux-x64/bin:" + os.environ["PATH"]
sh("npm install -g -s @playcanvas/splat-transform@3 && splat-transform --version")

# Compile gsplat's CUDA kernels now, so the training cell doesn't look frozen
sh("python -c 'from gsplat.cuda._backend import _C; assert _C is not None, \"gsplat CUDA build failed (is nvcc installed?)\"; print(\"gsplat CUDA kernels ready\")'", tail=80)

In [ ]:
%%writefile /kaggle/working/vm/frames.py
"""Turn a room video (or a folder of photos) into a clean set of sharp frames.

Usage:
    python frames.py INPUT OUT_DIR [--target 200] [--max-size 1600]

INPUT can be a video file, a folder containing one video, or a folder of photos
(jpg/png/heic). Frames are resized so the long side is at most --max-size and
saved as JPEGs named 00000.jpg, 00001.jpg, ... in capture order.
"""

import argparse
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import cv2
import numpy as np

VIDEO_EXT = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm"}
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp"}


def ffmpeg_exe() -> str:
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    import imageio_ffmpeg  # shipped with imageio[ffmpeg]

    return imageio_ffmpeg.get_ffmpeg_exe()


def probe_duration(video: Path) -> float:
    """Read the duration from ffmpeg's banner (no ffprobe needed)."""
    out = subprocess.run(
        [ffmpeg_exe(), "-hide_banner", "-i", str(video)],
        capture_output=True,
        text=True,
    ).stderr
    if "arib-std-b67" in out or "smpte2084" in out:
        print(
            "WARNING: this looks like an HDR (Dolby Vision / HLG) iPhone video. Colours may "
            "look washed out. Next time turn off Settings > Camera > Record Video > HDR Video."
        )
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", out)
    if not m:
        raise RuntimeError(f"Could not read video duration:\n{out[-2000:]}")
    h, mi, s = m.groups()
    return int(h) * 3600 + int(mi) * 60 + float(s)


def sharpness(path: Path) -> float:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    small = cv2.resize(img, (0, 0), fx=0.5, fy=0.5)
    return float(cv2.Laplacian(small, cv2.CV_64F).var())


def pick_sharpest(paths: list, target: int) -> list:
    """Split the sequence into `target` equal windows and keep the sharpest frame of each."""
    if len(paths) <= target:
        return paths
    windows = np.array_split(np.arange(len(paths)), target)
    return [paths[max(w, key=lambda i: sharpness(paths[i]))] for w in windows if len(w)]


def from_video(video: Path, out_dir: Path, target: int, max_size: int) -> int:
    duration = probe_duration(video)
    # Oversample 3x so we can drop blurry frames.
    fps = min(30.0, max(0.5, 3.0 * target / max(duration, 1e-3)))
    print(f"Video {video.name}: {duration:.1f}s, sampling at {fps:.2f} fps")
    with tempfile.TemporaryDirectory() as tmp:
        scale = f"scale='if(gt(iw,ih),min({max_size},iw),-2)':'if(gt(iw,ih),-2,min({max_size},ih))'"
        subprocess.run(
            [
                ffmpeg_exe(), "-hide_banner", "-loglevel", "error", "-i", str(video),
                "-vf", f"fps={fps},{scale}", "-q:v", "2", f"{tmp}/f_%06d.jpg",
            ],
            check=True,
        )
        raw = sorted(Path(tmp).glob("f_*.jpg"))
        print(f"Extracted {len(raw)} raw frames, keeping the sharpest {min(target, len(raw))}")
        keep = pick_sharpest(raw, target)
        for i, p in enumerate(keep):
            shutil.copy(p, out_dir / f"{i:05d}.jpg")
    return len(keep)


def load_image(path: Path):
    if path.suffix.lower() in {".heic", ".heif"}:
        from PIL import Image
        from pillow_heif import register_heif_opener

        register_heif_opener()
        return cv2.cvtColor(np.array(Image.open(path).convert("RGB")), cv2.COLOR_RGB2BGR)
    return cv2.imread(str(path), cv2.IMREAD_COLOR)


def from_images(images: list, out_dir: Path, target: int, max_size: int) -> int:
    images = sorted(images)
    if len(images) > target:
        images = pick_sharpest(images, target)
    n = 0
    for p in images:
        img = load_image(p)
        if img is None:
            print(f"  skipping unreadable {p.name}")
            continue
        h, w = img.shape[:2]
        s = min(1.0, max_size / max(h, w))
        if s < 1.0:
            img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(out_dir / f"{n:05d}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        n += 1
    return n


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("input")
    ap.add_argument("out_dir")
    ap.add_argument("--target", type=int, default=200)
    ap.add_argument("--max-size", type=int, default=1600)
    a = ap.parse_args()

    src, out_dir = Path(a.input), Path(a.out_dir)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    if src.is_file() and src.suffix.lower() in VIDEO_EXT:
        n = from_video(src, out_dir, a.target, a.max_size)
    elif src.is_dir():
        files = [p for p in src.rglob("*") if p.is_file()]
        videos = [p for p in files if p.suffix.lower() in VIDEO_EXT]
        images = [p for p in files if p.suffix.lower() in IMAGE_EXT]
        if images:
            print(f"Found {len(images)} photos in {src}")
            n = from_images(images, out_dir, a.target, a.max_size)
        elif videos:
            if len(videos) > 1:
                print(f"Found {len(videos)} videos, using the largest. Put one room per run.")
            n = from_video(max(videos, key=lambda p: p.stat().st_size), out_dir, a.target, a.max_size)
        else:
            sys.exit(f"No video or photos found in {src}")
    else:
        sys.exit(f"Input not found or unsupported: {src}")

    if n < 20:
        sys.exit(f"Only {n} frames. Need at least about 20 (ideally 100-300).")
    print(f"Wrote {n} frames to {out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/vm/sfm_align.py
"""Camera poses with COLMAP, then put the room in a game-friendly frame.

Usage:
    python sfm_align.py --images data/images --work work --out data/sparse/0
        [--matcher sequential|exhaustive] [--camera-height 1.4] [--known-scale S]

Output world frame (so splats, colliders and engines agree):
  * +Y is up, the floor is at y = 0
  * units are approximately metres. Scale is estimated by assuming you held the
    phone about --camera-height metres above the floor. Pass --known-scale to override.
  * walls are roughly aligned with the X and Z axes
Writes align.json next to --out with a seed point for the collider/player spawn.
"""

import argparse
import json
import shutil
from pathlib import Path

import numpy as np
import pycolmap


def rotation_between(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Smallest rotation taking unit vector a onto unit vector b."""
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    v, c = np.cross(a, b), float(np.dot(a, b))
    if c < -0.999999:  # opposite: rotate 180 deg about any perpendicular axis
        axis = np.cross(a, [1.0, 0, 0])
        if np.linalg.norm(axis) < 1e-6:
            axis = np.cross(a, [0, 0, 1.0])
        axis /= np.linalg.norm(axis)
        return 2 * np.outer(axis, axis) - np.eye(3)
    vx = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    return np.eye(3) + vx + vx @ vx / (1 + c)


def estimate_floor(y: np.ndarray, cam_y: float) -> float:
    """Floor height: a dense horizontal layer of points well below the cameras.

    If the floor was filmed, its points form a clear peak at the bottom of the
    height histogram. If not, fall back to a low percentile.
    """
    lo = np.percentile(y, 0.5)
    below = y[(y >= lo) & (y < cam_y - 0.4 * (cam_y - lo))]
    if len(below) > 200:
        hist, edges = np.histogram(below, bins=40)
        peak = int(np.argmax(hist))
        if hist[peak] > 4 * max(np.median(hist), 1):
            c = 0.5 * (edges[peak] + edges[peak + 1])
            w = edges[1] - edges[0]
            print("Floor: found a dense floor layer")
            return float(np.median(below[np.abs(below - c) <= w]))
    print("Floor: no clear floor layer (was the floor filmed?), using the lowest points")
    return float(np.percentile(y, 1))


def run_sfm(images: Path, work: Path, matcher: str) -> pycolmap.Reconstruction:
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    db = work / "database.db"

    reader = pycolmap.ImageReaderOptions()
    reader.camera_model = "SIMPLE_RADIAL"
    pycolmap.extract_features(
        db, images, camera_mode=pycolmap.CameraMode.SINGLE, reader_options=reader
    )
    if matcher == "sequential":
        pairing = pycolmap.SequentialPairingOptions()
        pairing.overlap = 20
        pairing.quadratic_overlap = True
        pycolmap.match_sequential(db, pairing_options=pairing)
    else:
        pycolmap.match_exhaustive(db)

    recs = pycolmap.incremental_mapping(db, images, work / "sparse")
    if not recs:
        raise SystemExit(
            "COLMAP could not reconstruct the scene. Usual causes: motion blur, blank "
            "walls, moving too fast, or too few overlapping frames."
        )
    best = max(recs.values(), key=lambda r: r.num_reg_images())
    n_img = len(list(images.iterdir()))
    print(
        f"Registered {best.num_reg_images()}/{n_img} images, {best.num_points3D()} points "
        f"({len(recs)} model(s))"
    )
    if best.num_reg_images() < 0.5 * n_img:
        print("WARNING: under half the frames registered. The splat will be missing parts of the room.")
    return best


def align(rec: pycolmap.Reconstruction, camera_height: float, known_scale):
    imgs = [rec.image(i) for i in rec.reg_image_ids()]
    centers = np.array([im.projection_center() for im in imgs])
    # OpenCV camera: +Y points down in the image, so camera "up" in world = R^T @ (0,-1,0)
    ups = np.array([im.cam_from_world().rotation.matrix().T @ np.array([0, -1.0, 0]) for im in imgs])
    up = ups.mean(0)
    R1 = rotation_between(up, np.array([0, 1.0, 0]))

    pts = np.array([p.xyz for p in rec.points3D.values() if p.track.length() >= 3])
    if len(pts) < 100:
        pts = np.array([p.xyz for p in rec.points3D.values()])
    pts_r, cams_r = pts @ R1.T, centers @ R1.T

    floor_y = estimate_floor(pts_r[:, 1], np.median(cams_r[:, 1]))
    cam_h = np.median(cams_r[:, 1]) - floor_y
    if known_scale:
        scale = float(known_scale)
    elif cam_h > 1e-6:
        scale = camera_height / cam_h
    else:
        print("WARNING: could not find the floor below the cameras; leaving scale at 1.")
        scale = 1.0

    # Yaw: line the dominant horizontal direction of the points up with +X.
    xz = pts_r[:, [0, 2]]
    lo, hi = np.percentile(xz, 5, axis=0), np.percentile(xz, 95, axis=0)
    inner = xz[np.all((xz >= lo) & (xz <= hi), axis=1)]
    evals, evecs = np.linalg.eigh(np.cov((inner - inner.mean(0)).T))
    major = evecs[:, np.argmax(evals)]
    yaw = -np.arctan2(major[1], major[0])  # rotate major axis (x,z) onto +X
    c, s = np.cos(yaw), np.sin(yaw)
    R2 = np.array([[c, 0, -s], [0, 1, 0], [s, 0, c]])
    R = R2 @ R1

    cams_R = centers @ R.T
    center_xz = np.median(cams_R[:, [0, 2]], axis=0)
    t = -scale * np.array([center_xz[0], floor_y, center_xz[1]])
    rec.transform(pycolmap.Sim3d(scale, pycolmap.Rotation3d(R), t))
    sim3 = {"scale": float(scale), "R": R.tolist(), "t": t.tolist()}  # new = scale * R @ old + t

    # Spawn/seed: the camera position nearest the middle of the capture path, at 1 m.
    cams = np.array([rec.image(i).projection_center() for i in rec.reg_image_ids()])
    mid = np.median(cams[:, [0, 2]], axis=0)
    nearest = cams[np.argmin(np.linalg.norm(cams[:, [0, 2]] - mid, axis=1))]
    pts_new = np.array([p.xyz for p in rec.points3D.values()])
    lo, hi = np.percentile(pts_new, 2, axis=0), np.percentile(pts_new, 98, axis=0)
    return {
        "scale_applied": scale,
        "scale_source": "known_scale" if known_scale else f"camera_height={camera_height}m",
        "seed": [float(nearest[0]), 1.0, float(nearest[2])],
        "camera_height_median": float(np.median(cams[:, 1])),
        "bounds_min": lo.tolist(),
        "bounds_max": hi.tolist(),
        "num_images": rec.num_reg_images(),
        "num_points": rec.num_points3D(),
        "sim3": sim3,
    }


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--images", required=True)
    ap.add_argument("--work", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--matcher", choices=["sequential", "exhaustive"], default="sequential")
    ap.add_argument("--camera-height", type=float, default=1.4)
    ap.add_argument("--known-scale", type=float, default=None)
    a = ap.parse_args()

    rec = run_sfm(Path(a.images), Path(a.work), a.matcher)
    info = align(rec, a.camera_height, a.known_scale)
    out = Path(a.out)
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    rec.write(out)
    (out.parent.parent / "align.json").write_text(json.dumps(info, indent=2))
    size = np.array(info["bounds_max"]) - np.array(info["bounds_min"])
    print(f"Room is roughly {size[0]:.1f} x {size[1]:.1f} x {size[2]:.1f} m (W x H x D)")
    print(json.dumps(info, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/vm/collider.py
"""Build a low-poly collision mesh (GLB) from a Gaussian-splat PLY.

Usage:
    python collider.py scene.ply collision.glb [--voxel 0.05] [--faces 30000]
        [--no-floor] [--align align.json]

Assumes the splat is in the frame written by sfm_align.py (+Y up, floor at y=0,
metres). It voxelises the solid Gaussians, runs marching cubes, decimates, and
adds a flat floor slab under the room, because floors are often textureless and
full of holes.
"""

import argparse
import json
from pathlib import Path

import numpy as np
import trimesh
from plyfile import PlyData
from scipy import ndimage
from skimage import measure


def load_solid_gaussians(path: Path, min_opacity: float, max_size: float) -> np.ndarray:
    v = PlyData.read(str(path))["vertex"].data
    xyz = np.stack([v["x"], v["y"], v["z"]], 1).astype(np.float64)
    opacity = 1 / (1 + np.exp(-np.asarray(v["opacity"], np.float64)))
    size = np.exp(np.max(np.stack([v["scale_0"], v["scale_1"], v["scale_2"]], 1), 1))
    keep = (opacity >= min_opacity) & (size <= max_size) & np.all(np.isfinite(xyz), 1)
    print(f"{keep.sum():,} of {len(xyz):,} Gaussians are solid enough for collision")
    return xyz[keep]


def build(xyz, voxel, min_count, bounds=None):
    if bounds is None:
        lo, hi = np.percentile(xyz, 0.5, 0), np.percentile(xyz, 99.5, 0)
    else:
        lo, hi = np.asarray(bounds[0]), np.asarray(bounds[1])
    pad = 3 * voxel
    lo, hi = lo - pad, hi + pad
    xyz = xyz[np.all((xyz >= lo) & (xyz <= hi), 1)]
    dims = np.ceil((hi - lo) / voxel).astype(int) + 1
    idx = np.floor((xyz - lo) / voxel).astype(int)
    grid = np.zeros(dims, np.int32)
    np.add.at(grid, tuple(idx.T), 1)
    occ = grid >= min_count
    if occ.sum() < 0.002 * occ.size and min_count > 1:
        print("Sparse splat: counting every voxel with at least one Gaussian as solid")
        occ = grid >= 1
    occ = ndimage.binary_closing(occ, iterations=1)  # seal pinholes
    labels, n = ndimage.label(occ)
    if n > 1:  # drop tiny floating islands
        sizes = ndimage.sum(occ, labels, range(1, n + 1))
        big = 1 + np.flatnonzero(sizes >= 20)
        if len(big):
            occ = np.isin(labels, big)
    if not occ.any():
        print("WARNING: no solid voxels found; the collider will only contain the floor")
        return None, lo, hi
    padded = np.pad(occ, 1).astype(np.float32)
    verts, faces, _, _ = measure.marching_cubes(padded, 0.5)
    verts = (verts - 1) * voxel + lo
    return trimesh.Trimesh(verts, faces, process=True), lo, hi


def floor_slab(lo, hi, thickness=0.1):
    size = [hi[0] - lo[0], thickness, hi[2] - lo[2]]
    box = trimesh.creation.box(extents=size)
    box.apply_translation([(lo[0] + hi[0]) / 2, -thickness / 2, (lo[2] + hi[2]) / 2])
    return box


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("ply")
    ap.add_argument("out")
    ap.add_argument("--voxel", type=float, default=0.05, help="voxel size in metres")
    ap.add_argument("--min-count", type=int, default=2, help="Gaussians per voxel to count as solid")
    ap.add_argument("--min-opacity", type=float, default=0.3)
    ap.add_argument("--max-size", type=float, default=0.25, help="ignore Gaussians bigger than this (m)")
    ap.add_argument("--faces", type=int, default=30000, help="target triangle count")
    ap.add_argument("--no-floor", action="store_true")
    ap.add_argument("--align", default=None, help="align.json from sfm_align.py (room bounds)")
    a = ap.parse_args()

    xyz = load_solid_gaussians(Path(a.ply), a.min_opacity, a.max_size)
    bounds = None
    if a.align and Path(a.align).exists():
        info = json.loads(Path(a.align).read_text())
        lo, hi = np.array(info["bounds_min"]), np.array(info["bounds_max"])
        margin = 0.15 * (hi - lo)
        bounds = (lo - margin, hi + margin)
    mesh, lo, hi = build(xyz, a.voxel, a.min_count, bounds)
    parts = []
    if mesh is not None:
        print(f"Marching cubes: {len(mesh.faces):,} triangles")
        if len(mesh.faces) > a.faces:
            mesh = mesh.simplify_quadric_decimation(face_count=a.faces)
            print(f"Decimated to {len(mesh.faces):,} triangles")
        parts.append(mesh)
    if not a.no_floor or not parts:
        parts.append(floor_slab(lo, hi))
    out = trimesh.util.concatenate(parts)
    out.export(a.out)
    print(f"Wrote {a.out}: {len(out.faces):,} triangles, extent {np.round(out.extents, 2)} m")


if __name__ == "__main__":
    main()

## 2. Video → sharp frames

In [ ]:
sh(f"python {VM}/frames.py '{INPUT_PATH}' {DATA}/images --target {TARGET_FRAMES} --max-size {MAX_IMAGE_SIZE}")
from PIL import Image
imgs = sorted((DATA / "images").glob("*.jpg"))
strip = Image.new("RGB", (6 * 320, 240))
for i, p in enumerate(imgs[:: max(1, len(imgs) // 6)][:6]):
    strip.paste(Image.open(p).resize((320, 240)), (i * 320, 0))
display(strip)

## 3. Camera poses (COLMAP) + level the floor + metric scale

In [ ]:
is_video = Path(INPUT_PATH).is_file() or not any(
    p.suffix.lower() in {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp"} for p in Path(INPUT_PATH).rglob("*"))
matcher = "sequential" if is_video else "exhaustive"
extra = f"--known-scale {KNOWN_SCALE}" if KNOWN_SCALE else ""
sh(f"python {VM}/sfm_align.py --images {DATA}/images --work {W}/colmap --out {DATA}/sparse/0 "
   f"--matcher {matcher} --camera-height {CAMERA_HEIGHT_M} {extra}", env={"PYTHONPATH": str(SFM_LIBS)})
ALIGN = json.loads((DATA / "align.json").read_text())

## 4. Train the Gaussian splat

In [ ]:
scaler = 1.0 if QUALITY == "full" else 0.25
shutil.rmtree(RES, ignore_errors=True)
sh(f"python simple_trainer.py default --data_dir {DATA} --data_factor 1 --result_dir {RES} "
   f"--no-normalize_world_space --save_ply --disable_viewer --steps_scaler {scaler} --test_every 100000",
   cwd=W / "gsplat" / "examples", env={"CUDA_VISIBLE_DEVICES": "0"}, tail=60)
plys = sorted(glob.glob(f"{RES}/ply/point_cloud_*.ply"), key=lambda p: int(Path(p).stem.split("_")[-1]))
PLY = plys[-1]
print("Final splat:", PLY, f"{os.path.getsize(PLY) / 1e6:.0f} MB")
renders = sorted(glob.glob(f"{RES}/renders/*.png"))
if renders:
    display(Image.open(renders[-1]))  # left: render, right: real photo

## 5. Export: compressed splats, HTML viewer, collision mesh

In [ ]:
shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()
shutil.copy(PLY, OUT / "scene.ply")
# Drop broken / near-invisible Gaussians, then write the compressed formats.
sh(f"splat-transform -w {OUT}/scene.ply --filter-nan --filter-value opacity,gt,0.02 {OUT}/clean.ply")
for ext in ("spz", "sog", "html"):  # falls back to the CPU encoder automatically when there's no WebGPU
    sh(f"splat-transform -w {OUT}/clean.ply {OUT}/scene.{ext}")
shutil.move(OUT / "clean.ply", OUT / "scene.ply")

seed = SEED_POS or ALIGN["seed"]
ALIGN["seed"] = seed
(OUT / "align.json").write_text(json.dumps(ALIGN, indent=2))
sh(f"python {VM}/collider.py {OUT}/scene.ply {OUT}/collision.glb --align {OUT}/align.json")

# Optional higher-quality collider from PlayCanvas (needs WebGPU/Vulkan; often missing on Kaggle)
try:
    sh(f"splat-transform -w {OUT}/scene.ply --seed-pos {seed[0]},{seed[1]},{seed[2]} "
       f"--voxel-external-fill --voxel-carve --collision-mesh {OUT}/scene.voxel.json", tail=8)
    print("Bonus: scene.collision.glb (PlayCanvas carve) written too")
except RuntimeError:
    print("(PlayCanvas voxel collider skipped: no WebGPU here. collision.glb from collider.py is fine.)")

zip_path = shutil.make_archive(str(W / f"viewmaxxing_{SCENE_NAME}"), "zip", OUT)
for f in sorted(OUT.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")
print("\nDownload:", zip_path, "(right panel → Output, or the file browser)")

## 6. What to do with it
- **See it now:** unzip, then double-click `scene.html` and drag to look around.
- **Blender 5.3+:** File → Import → **PLY** (`scene.ply`) or the **SPZ** importer (`scene.spz`). Set **Up axis = Y** in the import options, because this file is Y-up. Then File → Import → glTF → `collision.glb`, set it to invisible in renders, and use it for physics or nav.
- **Godot:** the `godot-gaussian-splatting` add-on loads the PLY. Add `collision.glb` as a `StaticBody3D` (or let the add-on generate collision itself).
- **Your character:** put `character.glb` at `align.json → seed` (it's in metres, on the floor, Y-up).

### Scale note
Scale is a guess based on `CAMERA_HEIGHT_M`. Check `align.json`: if a door that's really 2.0 m tall measures 2.3 m in Blender, set `KNOWN_SCALE = 2.0/2.3 * scale_applied` and re-run steps 3–5.

### If it looks bad
| Symptom | Fix |
|---|---|
| "Registered 40/200 images" | Blur or fast motion. Re-film slower with more overlap, or lower `TARGET_FRAMES` |
| Floating fog/blobs | Use `QUALITY="full"`, film more angles, crop in SuperSplat (free, superspl.at/editor) |
| CUDA out of memory | `MAX_IMAGE_SIZE = 1280` or `TARGET_FRAMES = 150` |
| Floor tilted / room sideways | You filmed mostly up or down. Film one loop holding the phone level |